1) Konzept in 3 Zeilen (damit der Kopf synchron ist)

Für Input $x \in \mathbb{R}^N$, weights $w \in \mathbb{R}^N$, bias $b \in \mathbb{R}$ :
- net input: $h=w^{\top} x+b$
- output: $y=f(h)$ (Step, Sigmoid, Tanh, Linear, ...)
- decision boundary: $w^{\top} x+b=0$ (Hyperplane)

Beziehung zur Tutoriums-Notation: $h=w^{\top} x-\theta \Rightarrow b=-\theta$.

In [2]:
import numpy as np

_stable_sigmoid(z)
Job: berechnet $\sigma(z)=\frac{1}{1+e^{-z}}$ numerisch stabil.
Warum: $\mathrm{np} . \exp ( \pm \mathrm{z})$ overflowt bei großen Beträgen (z. B. 800). Trick: Fallunterscheidung:
- für $z \geq 0: 1 /\left(1+e^{-z}\right)$ ist safe
- für $z<0$ : schreibe $\sigma(z)=\frac{e^z}{1+e^z}$, dann ist $\exp (z)$ klein statt riesig.

In [3]:
def _stable_sigmoid(z):
    """
    Numerisch stabile Sigmoid-Implementierung:
    verhindert overflow bei großen |z|.
    """
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z, dtype=float)

    pos = z >= 0
    neg = ~pos

    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[neg])
    out[neg] = ez / (1.0 + ez)
    return out

In [4]:
class ConnectionistNeuron:
    """
    Connectionist Neuron (Single Neuron) mit w, b und Aktivierungsfunktion.

    Unterstützte Aktivierungen:
      - "step": Heaviside (Perceptron) -> Ausgabe 0/1
      - "sigmoid": logistic -> Ausgabe (0,1)
      - "tanh": -> Ausgabe (-1,1)
      - "linear": Identität

    Training:
      - Für "step": Perceptron-Lernregel (online/stochastic)
      - Für "sigmoid": Gradient Descent auf Binary Cross-Entropy (logistic regression vibe)
    """

    def __init__(self, n_features, activation="step", lr=0.1, seed=None):
        self.n_features = int(n_features)
        self.activation_name = activation
        self.lr = float(lr)

        rng = np.random.default_rng(seed)
        # kleine Random-Init (symmetry breaking)
        self.w = rng.normal(loc=0.0, scale=0.01, size=self.n_features)
        self.b = 0.0

        self._set_activation(activation)

    def _set_activation(self, activation):
        if activation == "step":
            self.activation = lambda z: (z >= 0).astype(int)
            self._predict_proba = None  # keine "Proba" im strikten Sinn
        elif activation == "sigmoid":
            self.activation = _stable_sigmoid
            self._predict_proba = _stable_sigmoid
        elif activation == "tanh":
            self.activation = np.tanh
            self._predict_proba = None
        elif activation == "linear":
            self.activation = lambda z: z
            self._predict_proba = None
        else:
            raise ValueError(f"Unknown activation: {activation}")

    def net_input(self, X):
        """
        h = X @ w + b
        X kann shape (n_features,) oder (n_samples, n_features) haben.
        """
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def forward(self, X):
        """y = f(net_input)."""
        return self.activation(self.net_input(X))

    def predict(self, X, threshold=0.5):
        """
        Klassenvorhersage.
        - step: gibt bereits 0/1 aus
        - sigmoid: threshold auf probability-like output (default 0.5)
        """
        y = self.forward(X)
        if self.activation_name == "sigmoid":
            return (y >= threshold).astype(int)
        return y

    def predict_proba(self, X):
        """
        Probability-like output nur für sigmoid sinnvoll.
        """
        if self._predict_proba is None:
            raise RuntimeError("predict_proba ist nur für activation='sigmoid' verfügbar.")
        return self._predict_proba(self.net_input(X))

    def fit(self, X, y, epochs=20, shuffle=True, verbose=False):
        """
        Trainiert die Parameter (w,b).
        X: (n_samples, n_features)
        y: (n_samples,) mit Labels 0/1

        Returns: history (dict) mit z.B. loss.
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int).reshape(-1)

        if X.ndim != 2 or X.shape[1] != self.n_features:
            raise ValueError(f"X muss shape (n_samples, {self.n_features}) haben.")

        if self.activation_name == "step":
            return self._fit_perceptron(X, y, epochs, shuffle, verbose)

        if self.activation_name == "sigmoid":
            return self._fit_logistic_gd(X, y, epochs, verbose)

        raise RuntimeError("fit ist nur für activation='step' oder 'sigmoid' implementiert.")

    def _fit_perceptron(self, X, y, epochs, shuffle, verbose):
        """
        Perceptron update (online):
          y_hat = step(w·x + b)
          error = y - y_hat
          w += lr * error * x
          b += lr * error
        """
        rng = np.random.default_rng()
        history = {"errors_per_epoch": []}

        for ep in range(epochs):
            idx = np.arange(len(X))
            if shuffle:
                rng.shuffle(idx)

            errors = 0
            for i in idx:
                y_hat = int(self.forward(X[i]))
                error = int(y[i]) - y_hat
                if error != 0:
                    self.w += self.lr * error * X[i]
                    self.b += self.lr * error
                    errors += 1

            history["errors_per_epoch"].append(errors)
            if verbose:
                print(f"epoch {ep+1:>3}/{epochs}: misclassifications={errors}")

        return history

    def _fit_logistic_gd(self, X, y, epochs, verbose):
        """
        Batch Gradient Descent für logistic neuron:
          p = sigmoid(Xw + b)
          loss = BCE(y, p)
          grad_w = X^T (p - y) / n
          grad_b = mean(p - y)
        """
        history = {"loss": []}
        n = len(X)
        eps = 1e-12  # gegen log(0)

        for ep in range(epochs):
            z = self.net_input(X)
            p = _stable_sigmoid(z)

            # Binary Cross-Entropy
            p_clip = np.clip(p, eps, 1 - eps)
            loss = -np.mean(y * np.log(p_clip) + (1 - y) * np.log(1 - p_clip))
            history["loss"].append(float(loss))

            # Gradients
            diff = (p - y)  # shape (n,)
            grad_w = (X.T @ diff) / n
            grad_b = float(np.mean(diff))

            # Update
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            if verbose:
                print(f"epoch {ep+1:>3}/{epochs}: loss={loss:.6f}")

        return history


$\_\_\_\_$ init $\_\_\_\_$ (self, n_features, activation="step", lr=0.1, seed=None)

Job: Konstruktor. Legt Dimension fest und initialisiert Parameter.
- n_features : $N=$ Anzahl Eingangsfeatures (Dimension von $x$ )
- w : weight-vector $w \in \mathbb{R}^N$ (random klein initialisiert = symmetry breaking)
- b : bias $b \in \mathbb{R}$ (Start 0 )
- lr : learning rate $\eta$
- ruft _set_activation(...) auf, damit forward() weiß, welches $f$ gilt.

Warum random small: Wenn alles exakt 0 wäre, ist das Neuron zunächst komplett indifferent; bei mehreren Neuronen bricht man so Symmetrien (bei einem Neuron ist's weniger dramatisch, aber Standard).
```python
def __init__(self, n_features, activation="step", lr=0.1, seed=None):
      self.n_features = int(n_features)
      self.activation_name = activation
      self.lr = float(lr)

      rng = np.random.default_rng(seed)
      # kleine Random-Init (symmetry breaking)
      self.w = rng.normal(loc=0.0, scale=0.01, size=self.n_features) # Erwartungswert = 0 Standardabweichung = 0.01, Anzahl = n_features, gespeichert als np.array
      self.b = 0.0

      self._set_activation(activation) #TODO Was ist das? 

_set_activation(self, activation)
Job: mapped den String "step"/"sigmoid"/"tanh"/"linear" auf eine Callable self.activation(z).
- "step" : Heaviside $f(z)=1[z \geq 0] \rightarrow$ Output $0 / 1$
- "sigmoid" : logistic $\sigma(z) \in(0,1)$
- "tanh" : $\tanh (z) \in(-1,1)$
- "linear" : Identität $f(z)=z$

Zusätzlich: setzt self._predict_proba nur dann, wenn „probability-like“ Sinn macht (sigmoid).

Warum getrennt: Du willst ein Interface ( forward ) unabhängig von der Aktivierung.

```python self.activation = lambda z: (z >= 0).astype(int) ```
Das ist:

$$
f(z)= \begin{cases}1 & z \geq 0 \\ 0 & z<0\end{cases}
$$

also ein Hard Threshold.

self.activation ist eine Funktion. Der Rückgabewert hat stets die gleiche Form wie das Argument welches an sie übergeben wird. 

```python
def _set_activation(self, activation):
    if activation == "step":
        self.activation = lambda z: (z >= 0).astype(int)
        self._predict_proba = None  # keine "Proba" im strikten Sinn
    elif activation == "sigmoid":
        self.activation = _stable_sigmoid
        self._predict_proba = _stable_sigmoid
    elif activation == "tanh":
        self.activation = np.tanh
        self._predict_proba = None
    elif activation == "linear":
        self.activation = lambda z: z
        self._predict_proba = None
    else:
        raise ValueError(f"Unknown activation: {activation}")

```
net_input(self, x)
```


Job: berechnet den Total input (auch "pre-activation"):

$$
h=X w+b
$$

- Wenn x ein Vektor shape (n_features,) ist → ein Skalar $h$
- Wenn x eine Matrix (n_samples, n_features) ist → ein Vektor $h \in \mathbb{R}^{n \_ \text {samples }}$

Warum eigene Funktion: Das ist die zentrale lineare Stufe; praktisch für Debugging (z. B. checke erst $h$, dann $f(h)$ ).

```python
    def net_input(self, X):
        """
        h = X @ w + b
        X kann shape (n_features,) oder (n_samples, n_features) haben.
        """
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b


forward(self, x)
Job: führt den Forward pass aus:

$$
y=f(h)=f(X w+b)
$$


Das ist exakt "connectionist neuron".

Warum getrennt von predict: forward gibt die rohe Aktivierungsausgabe zurück; predict darf zusätzlich thresholding / casting machen.

```python 
def forward(self, X):
        """y = f(net_input)."""
        return self.activation(self.net_input(X))


predict(self, x, threshold=0.5)
Job: gibt Klassenentscheid zurück (meist 0/1).
- Bei "step" ist forward() bereits hart → direkt Return.
- Bei "sigmoid" liefert forward() Werte in (0,1). Dann wird via threshold binarisiert:

$$
\hat{y}=1[p \geq \tau]
$$


Standard: $\tau=0.5$.

Warum threshold als Parameter: Du kannst decision tradeoffs steuern (precision/recall vibes), ohne neu zu trainieren.

```python

def predict(self, X, threshold=0.5):
        """
        Klassenvorhersage.
        - step: gibt bereits 0/1 aus
        - sigmoid: threshold auf probability-like output (default 0.5)
        """
        y = self.forward(X)
        if self.activation_name == "sigmoid":
            return (y >= threshold).astype(int)
        return y


predict_proba(self, x)
Job: gibt probability-like Output $p=\sigma(X w+b)$ zurück (nur sigmoid).
Wirft Fehler, wenn Aktivierung nicht sigmoid ist.
Warum separate Funktion: Semantisch ist "proba" nur sinnvoll, wenn Output im (0,1)Intervall interpretiert wird.

```python

   def predict_proba(self, X):
        """
        Probability-like output nur für sigmoid sinnvoll.
        """
        if self._predict_proba is None:
            raise RuntimeError("predict_proba ist nur für activation='sigmoid' verfügbar.")
        return self._predict_proba(self.net_input(X))**

```
fit(self, x, y, epochs=20, shuffle=True, verbose=False)
```


Job: Training wrapper: validiert Shapes und routet zum passenden Lernalgorithmus:
- "step" → _fit_perceptron(...)
- "sigmoid" → _fit_logistic_gd(...)

Inputs:
- x : Matrix (n_samples, n_features)
- y : Labels $\left(n \_\right.$samples,$)$als $0 / 1$

Warum Wrapper: einheitliches API, egal ob du Perceptron oder Logistic neuron trainierst.__init__

```python
def fit(self, X, y, epochs=20, shuffle=True, verbose=False):
        """
        Trainiert die Parameter (w,b).
        X: (n_samples, n_features)
        y: (n_samples,) mit Labels 0/1

        Returns: history (dict) mit z.B. loss.
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int).reshape(-1)

        if X.ndim != 2 or X.shape[1] != self.n_features:
            raise ValueError(f"X muss shape (n_samples, {self.n_features}) haben.")

        if self.activation_name == "step":
            return self._fit_perceptron(X, y, epochs, shuffle, verbose)

        if self.activation_name == "sigmoid":
            return self._fit_logistic_gd(X, y, epochs, verbose)

        raise RuntimeError("fit ist nur für activation='step' oder 'sigmoid' implementiert.")

_fit_perceptron(self, x, y, epochs, shuffle, verbose)
Job: klassisches Perceptron learning (online/stochastic updates).
Ablauf pro Sample $x_i$ :
1. Vorhersage $\hat{y}_i=\operatorname{step}\left(w^{\top} x_i+b\right)$
2. Fehler $e_i=y_i-\hat{y}_i$ (liegt in $\{-1,0,1\}$ )
3. Update:

$$
w \leftarrow w+\eta e_i x_i, \quad b \leftarrow b+\eta e_i
$$


Intuition:
- Wenn ein positives Beispiel als 0 klassifiziert wurde $(e=+1)$, schiebst du $w$ Richtung $x$.
- Wenn ein negatives Beispiel fälschlich als 1 klassifiziert wurde $(e=-1)$, schiebst du $w$ weg von $x$.

Output: history["errors_per_epoch"] = wie viele Samples pro Epoche falsch waren.
Limit: funktioniert nur zuverlässig bei linear separable Daten (sonst oszilliert's).

```python
def _fit_perceptron(self, X, y, epochs, shuffle, verbose):
        """
        Perceptron update (online):
          y_hat = step(w·x + b)
          error = y - y_hat
          w += lr * error * x
          b += lr * error
        """
        rng = np.random.default_rng()
        history = {"errors_per_epoch": []}

        for ep in range(epochs):
            idx = np.arange(len(X))
            if shuffle:
                rng.shuffle(idx)

            errors = 0
            for i in idx:
                y_hat = int(self.forward(X[i]))
                error = int(y[i]) - y_hat
                if error != 0:
                    self.w += self.lr * error * X[i]
                    self.b += self.lr * error
                    errors += 1

            history["errors_per_epoch"].append(errors)
            if verbose:
                print(f"epoch {ep+1:>3}/{epochs}: misclassifications={errors}")

        return history

 
 _fit_logistic_gd(self, x, y, epochs, verbose)
Job: Batch Gradient Descent für ein sigmoid neuron (das ist praktisch logistic regression als Single-Neuron-Netz).
1. Compute logits $z=X w+b$
2. Probabilities $p=\sigma(z)$
3. Loss: Binary Cross-Entropy (BCE)

$$
\mathcal{L}=-\frac{1}{n} \sum_i\left[y_i \log \left(p_i\right)+\left(1-y_i\right) \log \left(1-p_i\right)\right]
$$

4. Gradients (klassisch, elegant):

$$
\nabla_w=\frac{1}{n} X^{\top}(p-y), \quad \nabla_b=\operatorname{mean}(p-y)
$$

5. Update:

$$
w \leftarrow w-\eta \nabla_w, \quad b \leftarrow b-\eta \nabla_b
$$


Warum BCE statt "MSE": BCE passt zur Bernoulli-Interpretation und hat bessere Gradienten-Eigenschaften bei Klassifikation.

Output: history["loss"] pro Epoche.
 
 
 ```python 
 def _fit_logistic_gd(self, X, y, epochs, verbose):
        """
        Batch Gradient Descent für logistic neuron:
          p = sigmoid(Xw + b)
          loss = BCE(y, p)
          grad_w = X^T (p - y) / n
          grad_b = mean(p - y)
        """
        history = {"loss": []}
        n = len(X)
        eps = 1e-12  # gegen log(0)

        for ep in range(epochs):
            z = self.net_input(X)
            p = _stable_sigmoid(z)

            # Binary Cross-Entropy
            p_clip = np.clip(p, eps, 1 - eps)
            loss = -np.mean(y * np.log(p_clip) + (1 - y) * np.log(1 - p_clip))
            history["loss"].append(float(loss))

            # Gradients
            diff = (p - y)  # shape (n,)
            grad_w = (X.T @ diff) / n
            grad_b = float(np.mean(diff))

            # Update
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            if verbose:
                print(f"epoch {ep+1:>3}/{epochs}: loss={loss:.6f}")

        return history


In [5]:
# AND dataset
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y = np.array([0, 0, 0, 1], dtype=int)

neu = ConnectionistNeuron(n_features=2, activation="step", lr=0.2, seed=1)
hist = neu.fit(X, y, epochs=20, verbose=True)

print("w =", neu.w, "b =", neu.b)
print("pred =", neu.predict(X))


epoch   1/20: misclassifications=1
epoch   2/20: misclassifications=2
epoch   3/20: misclassifications=2
epoch   4/20: misclassifications=2
epoch   5/20: misclassifications=3
epoch   6/20: misclassifications=1
epoch   7/20: misclassifications=1
epoch   8/20: misclassifications=1
epoch   9/20: misclassifications=0
epoch  10/20: misclassifications=0
epoch  11/20: misclassifications=0
epoch  12/20: misclassifications=0
epoch  13/20: misclassifications=0
epoch  14/20: misclassifications=0
epoch  15/20: misclassifications=0
epoch  16/20: misclassifications=0
epoch  17/20: misclassifications=0
epoch  18/20: misclassifications=0
epoch  19/20: misclassifications=0
epoch  20/20: misclassifications=0
w = [0.20345584 0.40821618] b = -0.6000000000000001
pred = [0 0 0 1]


In [6]:
neu2 = ConnectionistNeuron(n_features=2, activation="sigmoid", lr=0.5, seed=1)
hist2 = neu2.fit(X, y, epochs=200, verbose=False)

print("proba =", neu2.predict_proba(X).round(3))
print("pred  =", neu2.predict(X))
print("last loss =", hist2["loss"][-1])


proba = [0.008 0.149 0.149 0.787]
pred  = [0 0 0 1]
last loss = 0.14356898214087035
